## Extracción de información de la página del Rues con autenticación

<p><b>TIP:</b> Para descomentar una línea de código en Visual Studio Code, puede señalar la línea que se quiera descomentar y utilizar el comando <b>'Ctrl + k + u'</b>. Por el contrario, para comentar una línea puede usar el comando <b>'Ctrl + k + c'</b>. En caso de ya tenerlos instalados no se debe hacer nada.</p><p><b>NOTA:</b> Se debe descargar <b>Chrome Driver</b> que es el navegador de prueba que permitirá ejecutar la automatización, se recomienda instalarlo en el escritorio ya que el código está para que lo ejecute ahí. Puede descargar Chrome Driver en el siguiente link: <a href='URL'>https://developer.chrome.com/docs/chromedriver/downloads?hl=es-419</a>
</p>

<p>Librerías a cargar, debemos cargar <b>Selenium</b> (automatización de procesos), <b>Beautifulsoup</b> (web scrapping), <b>pandas</b>, <b>time</b> y <b>pathlib</b> (generalizar rutas).</p> El bloque de abajo detecta automáticamente si ya se tienen las librerías necesarias. En caso de no tenerlas instaladas este mismo las instalará automáticamente

In [1]:
import subprocess
import sys
# Lista de librerías requeridas
required_packages = [
    ("pandas", "pandas"),
    ("beautifulsoup4", "bs4"),
    ("selenium", "selenium"),
    ("time", "time"),
    ("pathlib", "pathlib")
]

def install_package(package_name):
    """Instala un paquete usando pip."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

def check_and_install_packages(packages):
    """Comprueba si los paquetes están instalados, y los instala si no lo están."""
    for package_name, module_name in packages:
        try:
            __import__(module_name)
            print(f"{package_name} ya está instalado.")
        except ImportError:
            print(f"{package_name} no está instalado. Instalando...")
            install_package(package_name)

# Comprobar e instalar los paquetes necesarios
check_and_install_packages(required_packages)

# Ejemplo de uso de las librerías (para verificar que se instalaron correctamente)
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementNotInteractableException
from bs4 import BeautifulSoup
import pandas as pd
import time
from pathlib import Path

print("Todas las librerías están instaladas y listas para usar.")

pandas ya está instalado.
beautifulsoup4 ya está instalado.
selenium ya está instalado.
time ya está instalado.
pathlib ya está instalado.
Todas las librerías están instaladas y listas para usar.


<p><b>Ingresar los NITs a extraer información</b>. Si los tiene en un archivo de excel se pueden cargar con lel siguiente bloque de código abajo, solo debe especificar la ruta donde está el archivo y luego especificar el nombre de la columna donde están los nits en el archivo.</p>

In [3]:
nits = pd.read_excel(r"C:\Users\jeraso\OneDrive - CAMARA DE COMERCIO DE CALI\Analítica y Estudios Económicos\Documentos_generales\4.RANKINGS_700+\1.Rankings_2024\1.BD_Preliminar\13.Versión_final\20241002_RANKINGS_VF.xlsx",
                     sheet_name='ranking')['NIT'].iloc[10:21].astype(int) # Aquí se deben cargar los NITs a los cuales se les quiera extraer la información. 

# Imprimir la lista
print(len(nits))

11


**Iniciar sesión**. Se debe ingresar el usuario y contraseña

In [4]:
# Credenciales de login
username = 'bmunoz@ccc.org.co'
password = 'Camara2025*'

Abrir el chromeDriver. Aquí se debe actualizar la ruta donde se haya instalado el ChromeDriver. Por recomendación, lo mejor es instalarlo en la aplicación de escritorio (el usuario se llama practeec, en caso de que ya no sea así debe cambiarlo). Además iniciaremos sesión automaticamente, aunque también se puede hacer de forma manual (si el usuario y/o contraseña cambia, deben cambairse estos valores en el código)

In [13]:
# Ruta al ejecutable de ChromeDriver (ajusta esta ruta según donde hayas descargado el driver)
chrome_driver_path = Path.home() / 'Downloads' / 'Chromedriver.exe'
# Crear una instancia del servicio de ChromeDriver
service = Service(chrome_driver_path)
# Inicializar el navegador usando el servicio
driver = webdriver.Chrome(service=service)
# URL de la página de login
login_url = "https://www.rues.org.co/?old=true"

# Navegar a la página de login
driver.get(login_url)

# Darle tiempo (5 segundos)
time.sleep(5)

# Botón cerrar emergente
xpath_boton_cerrar_emergente = "//button[@class='btn btn-success btn-sm']"
boton_cerrar_emergente = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.XPATH, xpath_boton_cerrar_emergente)))
driver.execute_script("arguments[0].scrollIntoView();", boton_cerrar_emergente)
boton_cerrar_emergente.click()

#Hacer click en el botón de acceso
WebDriverWait(driver, 15).until(EC.element_to_be_clickable((By.CLASS_NAME, 'login-call'))).click()

# Encontrar los campos de login y enviar las credenciales
try:
    email_element = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.ID, 'Email')))
    password_element = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.ID, 'Password')))
    
    # Desplazarse a los elementos para asegurarse de que están visibles
    driver.execute_script("arguments[0].scrollIntoView();", email_element)
    driver.execute_script("arguments[0].scrollIntoView();", password_element)
    
    email_element.send_keys(username)
    password_element.send_keys(password)
except (TimeoutException, ElementNotInteractableException) as e:
    print("No se pudo interactuar con los campos de login:", e)
    driver.quit()
    raise

# Encontrar el botón por su texto y hacer clic
driver.find_element(By.CLASS_NAME, 'btn-blue').click() # Ajusta según el botón de submit

In [ ]:
# Ciclo for para los n NITs enlistados
results = []

def dar_click_boton_regresar():
    boton_regresar_xpath = "//a[@class='btn-gt']"
    ver_boton_regresar = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH, boton_regresar_xpath)))
    WebDriverWait(driver, 20).until(EC.visibility_of(ver_boton_regresar))
    WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.XPATH, boton_regresar_xpath)))
    driver.execute_script("arguments[0].scrollIntoView(true);", ver_boton_regresar)
    driver.execute_script("arguments[0].click();", ver_boton_regresar)


############## CICLO FOR PARA LOS N NITS ####################

# Matriz a rellenar más adelante
data = {
    'NIT': None,
    'CIUU': None
}

for nit in nits:
   
    # Esperar hasta que el campo de "Número de identificación" esté presente y realizar la consulta
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'txtNIT')))
    driver.find_element(By.ID, 'txtNIT').clear()
    driver.find_element(By.ID, 'txtNIT').send_keys(nit)
    driver.find_element(By.ID, "btnConsultaNIT").click()

    # Esperar hasta que el botón esté presente y hacer clic
    
    # try:
    # Encontrar la celda que contiene el texto "ACTIVA"
    activa_td_xpath = "//table[@id='rmTable2']//tbody//tr//td[text()='ACTIVA' or text()='ACTIVA, CONSTITUCIÓN POR TRASLADO']"
    activa_td = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, activa_td_xpath))
    )

    # Encontrar la fila padre de la celda encontrada
    parent_row = activa_td.find_element(By.XPATH, "./ancestor::tr")
    # Encontrar la celda con tabindex="0" en la misma fila
    target_td = parent_row.find_element(By.XPATH, ".//td[@tabindex='0']")
    # Esperar a que la celda con tabindex="0" sea clicable
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, ".//td[@tabindex='0']"))
    )
    # Hacer clic en la celda con tabindex="0"
    target_td.click()

    # Dar click en el link de info
    encontrar_link = WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.LINK_TEXT, "nfo")))
    encontrar_link.click()

    # Esperar hasta que la tabla esté presente
    div_xpath = "//div[@class='card-body']//ul[@class='cleanlist']"
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, div_xpath)))

    # Extraer el contenido de la página
    page_source = driver.page_source

    # Analizar el contenido con BeautifulSoup
    soup = BeautifulSoup(page_source, 'html.parser')

    # Buscar la tabla con datos
    ul_list = soup.select_one('ul.cleanlist')
    data = {'NIT': nit}
    if ul_list:
        # Concatenar los elementos de la lista en una sola cadena
        resultado = ', '.join([li.find('b').text.strip() for li in ul_list.find_all('li') if li.find('b')])
        # Agregar el resultado al diccionario
        data['CIUU'] = resultado
    else:
        data['CIUU'] = None

    # Agregar los datos del NIT actual a la lista de resultados
    results.append(data)
    dar_click_boton_regresar()

if results:
    # Convertir los resultados a un DataFrame de pandas para facilitar la manipulación
    df = pd.DataFrame(results)

    # Ruta carpeta para guardar el archivo
    rues_descarga_path = Path.home() / 'Downloads' / 'CIUU_rues_2025.xlsx'
    # Crear el archivo excel 
    df.to_excel(rues_descarga_path, index=False)

# Cerrar el navegador
driver.quit()

print(df.info())

In [ ]:
# df = pd.DataFrame(results)
# df.to_excel(rues_descarga_path, index=False)